# FINM 32000: Homework 2

By Andrew McLaughlin

## Problem 1

Consider a particular stock index that is defined to have value equal to the price of a fixed basket of non-dividend-paying stocks. Suppose that it follows the Black-Scholes dynamics with $\sigma = 0.4$, and that the time-0 index level is $S_0 = 100$. Consider a three-month ($=0.25$ year) up-and-out European put, struck at 95, with a discretely monitored knock-out barrier at 114, observed at times $0.02, 0.04, \ldots, 0.24$. That is, our option knocks out if and only if the index is at or above 114 at an observation time (where the unit of time in this course will always be years, unless otherwise indicated). Let the constant risk-free interest rate be $r = 0$.

In [20]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import bisect, brentq
from copy import copy

In [21]:
class UpAndOutPut:

    def __init__(self, K, T, barrier, observationinterval):
        self.K = K
        self.T = T
        self.barrier = barrier
        self.observationinterval = observationinterval

In [22]:
#note the template notebook had the barrier at 107 but the homework pdf requests 114
hw2contract = UpAndOutPut(K=95, T=0.25, barrier=114, observationinterval=0.02)

In [23]:
#hwk 1 GBMdyanmics, changed X to S to reflect hw2dynamics
class GBMdynamics:

    def __init__(self, S, r, rGrow, sigma=None):
        self.S = S
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self

In [24]:
# Use the same GBMdynamics class from HW 1

hw2dynamics = GBMdynamics(S=100, sigma=0.4, rGrow=0, r=0)
hw2dynamics

### (a)

Write a Python function to price our option at time 0 using a trinomial tree with probabilities specified in L2.21. Some of the code is already provided for you.

The provided code uses the time step $\Delta t = T/N$ as suggested in class. We will want the barrier-monitoring times to be represented in the tree, preferably without introducing unequal time intervals anywhere, so we will want to choose $N$ a multiple of 25. Your code should be able to accept any such $N$; a user who desires high accuracy can choose $N$ large; a user who desires high speed can choose $N$ small. For FINM 32000, please report a price using $N$ chosen large enough that your output has converged, in your judgment (no proof needed), to within $0.01 of the true price. In this example, $N = 100$ will not be sufficient.

The provided code chooses the space step $\Delta x$ such that the log of the barrier level $H = 114$ is exactly halfway between consecutive log price levels of the tree. Subject to this constraint, it chooses $\Delta x$ close to the recommended $\sigma \sqrt{3\Delta t}$ value. In other words, the constraint is that there exists an integer $j$ such that $\log(114)$ is halfway between the $j$th and $(j+1)$th log-price levels:

$$
\log S_0 + (j + 0.5)\Delta x = \log H.
$$

And the integer $j$ is chosen such that the $\Delta x$ which satisfies the constraint is approximately $\sigma \sqrt{3\Delta t}$, so we take $j$ to be the nearest integer to

$$
\frac{\log(H/S_0)}{\sigma \sqrt{3\Delta t}} - 0.5.
$$

Why do we have this "halfway between" requirement? If you try instead putting $\log H$ at a log-price level, you will find the accuracy to be worse than the "halfway between" procedure, for this discretely monitored barrier option.

In [34]:
class TreeEngine:

    def __init__(self, N):
        self.N = N

    def price_upandout(self, dynamics, contract):

        deltat = contract.T / self.N
        J = np.ceil(np.log(contract.barrier/dynamics.S)/(dynamics.sigma*np.sqrt(3*deltat))-0.5)
        deltax = np.log(contract.barrier/dynamics.S)/(J+0.5)

        Sgrid = dynamics.S*np.exp(np.linspace(self.N, -self.N, num=2*self.N+1, endpoint=True)*deltax)
        #Here I decided to make the SMALLER indexes in this array correspond to HIGHER S

        numTimestepsPerObs = contract.observationinterval/deltat
        if abs(numTimestepsPerObs-round(numTimestepsPerObs)) > 1e-8:
            raise ValueError("This value of N fails to place the observation dates in the tree.")

        nu =  dynamics.rGrow - dynamics.sigma**2 * .5      # complete this

        A = (dynamics.sigma**2 * deltat + nu**2 * deltat**2) / deltax**2
        B = nu * deltat / deltax


        Pu = .5 * (A + B)    # complete this
        Pd = .5 * (A - B)    # complete this
        Pm = 1 - A           # complete this

        optionprice = np.maximum(contract.K-Sgrid,0)   #an array of time-T option prices.

        #Next, induct backwards to time 0, updating the optionprice array
        #Hint: if x is an array, then what are x[2:] and x[1:-1] and x[:-2]

        for t in np.linspace(self.N-1, 0, num=self.N, endpoint=True)*deltat:
            # insert lines of code here if needed
            C_u = optionprice[:-2]
            C_m = optionprice[1:-1]
            C_d = optionprice[2:]
            optionprice = np.exp(-1 * dynamics.r * deltat) * (Pu * C_u + Pm * C_m + Pd * C_d) #complete this
            Sgrid = Sgrid[1:-1]
            step = int(round(t / deltat))
            if step % int(round(numTimestepsPerObs)) == 0 and step != 0:
                optionprice[Sgrid >= contract.barrier] = 0

        return optionprice[0]
        #The [0] is assuming that we are shrinking the optionprice array in each iteration of the loop,
        #until finally there is only 1 element in the array.
        #If instead you are keeping unchanged the size of the optionprice array in each iteration,
        #then you need to change the [0] to a different index.

    def price_euro(self, dynamics, contract):
        
        deltat = contract.T / self.N
        J = np.ceil(np.log(contract.barrier/dynamics.S)/(dynamics.sigma*np.sqrt(3*deltat))-0.5)
        deltax = np.log(contract.barrier/dynamics.S)/(J+0.5)

        Sgrid = dynamics.S*np.exp(np.linspace(self.N, -self.N, num=2*self.N+1, endpoint=True)*deltax)
        #Here I decided to make the SMALLER indexes in this array correspond to HIGHER S

        numTimestepsPerObs = contract.observationinterval/deltat
        if abs(numTimestepsPerObs-round(numTimestepsPerObs)) > 1e-8:
            raise ValueError("This value of N fails to place the observation dates in the tree.")

        nu =  dynamics.rGrow - dynamics.sigma**2 * .5      # complete this

        A = (dynamics.sigma**2 * deltat + nu**2 * deltat**2) / deltax**2
        B = nu * deltat / deltax


        Pu = .5 * (A + B)    # complete this
        Pd = .5 * (A - B)    # complete this
        Pm = 1 - A           # complete this

        optionprice = np.maximum(contract.K-Sgrid,0)   #an array of time-T option prices.

        #Next, induct backwards to time 0, updating the optionprice array
        #Hint: if x is an array, then what are x[2:] and x[1:-1] and x[:-2]

        for t in np.linspace(self.N-1, 0, num=self.N, endpoint=True)*deltat:
            # insert lines of code here if needed
            C_u = optionprice[:-2]
            C_m = optionprice[1:-1]
            C_d = optionprice[2:]
            optionprice = np.exp(-1 * dynamics.r * deltat) * (Pu * C_u + Pm * C_m + Pd * C_d) #complete this
        #    Sgrid = Sgrid[1:-1]
        #    step = int(round(t / deltat))
        #    if step % int(round(numTimestepsPerObs)) == 0 and step != 0:
        #        optionprice[Sgrid >= contract.barrier] = 0

        return optionprice[0]  


In [26]:
hw2tree=TreeEngine(N=100)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.311979858729038)

In [27]:
hw2tree=TreeEngine(N=500)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.304324358395272)

In [ ]:
hw2tree=TreeEngine(N=1000)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.301546106266766)

In [29]:
hw2tree=TreeEngine(N=10000)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.301072607079835)

After testing multiple Ns, N = 1000 provided good accuracy up to .01 cents compared to N =10,000 which only improved the result by ten-thousanths of a decimal.


Regarding the "halfway between" requirement: 

Because the option value is discontinuous at the barrier, placing the barrier exactly on a tree node forces that node to be treated as fully knocked out, even though the node is only an approximation to a range of nearby outcomes. That creates an asymmetric approximation error near the barrier. By placing the barrier halfway between two adjacent log-price nodes, one node lies clearly below the barrier and the next lies clearly above it, so the discontinuity is represented between nodes rather than on a node. This gives a more balanced and typically more accurate approximation of the barrier option’s value.

### (b)

Consider an up-and-in put with the same terms. Specifically, this option has the same strike, expiry, barrier, and monitoring dates, but it pays at expiry the put payoff only if the index was at or above the 114 knock-in barrier at some monitoring date; otherwise, it pays nothing.

Using your part (a) result, find the time-0 price of the up-and-in put.

In [35]:
#If we know the price of a vanilla-put and understand an up and in put + an up and out put = vanilla put, then we can solve

tree_upandout=TreeEngine(N=1000)

price_upandout = tree_upandout.price_upandout(hw2dynamics, hw2contract)
price_euro = tree_upandout.price_euro(hw2dynamics, hw2contract)

price_upandin = price_euro - price_upandout

price_upandin


np.float64(0.217341132747511)

### (c)

Consider a continuously monitored barrier option paying at time $T = 0.25$ the amount

$$
(95 - S_{0.25})^+ \mathbf{1}_{\max_{0 \le t \le 0.25} S_t < 114},
$$

where the indicator variable $\mathbf{1}(A) := \mathbf{1}_A := 1$ if event $A$ occurs, 0 otherwise.

#### (c1)

Is the time-0 price of the continuously monitored barrier option greater than or smaller than the time-0 price of the discretely monitored option in (a)? Justify briefly without doing any numerical calculations. One sentence is enough.

-  Logically, I would say the continuously monitored option must be less than the discretely monitored option because if the underlying hit the barrier between discrete monitoring periods but settled below the barrer before the next observation then the discretely monitored option would still have value and the continiously monitored would not. Overall, any path that survives under continuous must survive under discrete, but not vice versa.

#### (c2)

The continuously monitored barrier option can be replicated by a portfolio of $T$-expiry options, long 1 plain vanilla put struck at 95, and short $\alpha$ plain vanilla calls struck at 136.8.

The replication strategy is as follows. If $S$ does not hit the barrier before time $T$, then simply collect the time-$T$ payout of the 95 put, as desired. If $S$ does hit the barrier, then at the time when $S$ is at the barrier, the 1 unit of the vanilla put has value that exactly cancels the value of the $-\alpha$ units of the plain vanilla call; so at that time, we close out the portfolio positions, for a net payment of zero, as desired.

Solve analytically for the quantity $\alpha$ that makes this replication strategy valid, and find the time-0 value of the continuously monitored barrier option. Do not use a tree.